# NWB Production Schema Demo

This notebook demonstrates how to import and use the NWB Production schema from the U19 pipeline.

The NWB Production schema provides tables for:
- Managing NWB export jobs
- Tracking job status through the pipeline
- Logging status transitions
- Storing validation results

## 1. Import Required Modules

First, import the necessary modules including DataJoint and the U19 pipeline.

In [1]:
# from scripts.conf_file_finding import try_find_conf_file

# try_find_conf_file()

import datajoint as dj
import pandas as pd
from datetime import datetime

# Import the NWB production schema
from u19_pipeline import nwb_production

print("✅ NWB Production schema imported successfully!")
print(f"Schema name: {nwb_production.schema.database}")

[2026-01-17 13:18:24,331][INFO]: DataJoint 0.14.4 connected to ct5868@datajoint00.pni.princeton.edu:3306


✅ NWB Production schema imported successfully!
Schema name: u19_nwb_production


## 2. Explore NWB Export Status Table

The `NwbExportStatus` table is a lookup table containing the possible status values for NWB export jobs.

In [ ]:
# Display the table structure
nwb_production.NwbExportStatus()

In [ ]:
# View all status values
status_df = pd.DataFrame(nwb_production.NwbExportStatus.fetch())
print("Available NWB Export Statuses:\n")
print(status_df.to_string(index=False))

## 3. Explore NWB Export Job Table

The `NwbExportJob` table stores information about each NWB export job, including the subject, session, and current status.

In [ ]:
# Display the NwbExportJob table structure
print("NwbExportJob Table Definition:\n")
print(nwb_production.NwbExportJob.describe())
print("\n" + "="*80 + "\n")

# Check how many jobs exist
job_count = len(nwb_production.NwbExportJob())
print(f"Total NWB export jobs in database: {job_count}")

## 4. Explore Part Tables

The NwbExportJob table has three part tables for different modalities:
- **BehaviorExport**: Tracks behavior data exports
- **EphysExport**: Tracks electrophysiology data exports  
- **ImagingExport**: Tracks imaging data exports

In [ ]:
# Display part table structures
print("BehaviorExport Table:\n")
print(nwb_production.NwbExportJob.BehaviorExport.describe())
print("\n" + "="*80 + "\n")

print("EphysExport Table:\n")
print(nwb_production.NwbExportJob.EphysExport.describe())
print("\n" + "="*80 + "\n")

print("ImagingExport Table:\n")
print(nwb_production.NwbExportJob.ImagingExport.describe())

## 5. Explore Log and Validation Tables

The schema also includes tables for logging status changes and storing validation results.

In [ ]:
# NwbExportLogStatus - tracks all status transitions
print("NwbExportLogStatus Table:\n")
print(nwb_production.NwbExportLogStatus.describe())
print("\n" + "="*80 + "\n")

# NwbExportValidation - stores validation results
print("NwbExportValidation Table:\n")
print(nwb_production.NwbExportValidation.describe())

## 6. View ERD (Entity Relationship Diagram)

Visualize the relationships between all tables in the NWB Production schema.

In [ ]:
# Draw the Entity Relationship Diagram
dj.ERD(nwb_production)

## 7. Query Example: View Jobs by Status

Let's see how to query jobs by their current status.

In [ ]:
# Query jobs with specific statuses
# Status IDs: 0=QUEUED, 1=DATA_VALIDATION, 2=PROCESSING, 3=COMPLETED, -1=FAILED

# Example: Get all active jobs (not completed or failed)
active_jobs = nwb_production.NwbExportJob & 'status_nwb_id >= 0 AND status_nwb_id < 3'
print(f"Active jobs: {len(active_jobs)}")

# Example: Get all completed jobs
completed_jobs = nwb_production.NwbExportJob & 'status_nwb_id = 3'
print(f"Completed jobs: {len(completed_jobs)}")

# Example: Get all failed jobs
failed_jobs = nwb_production.NwbExportJob & 'status_nwb_id = -1'
print(f"Failed jobs: {len(failed_jobs)}")

## 8. Summary

The NWB Production schema provides a complete framework for managing NWB export jobs:

- **Job Tracking**: Submit jobs with modality selection (behavior, ephys, imaging)
- **Status Management**: Track jobs through pipeline stages (QUEUED → DATA_VALIDATION → PROCESSING → COMPLETED)
- **Audit Trail**: Complete history of status transitions with timestamps and error messages
- **Validation**: Store validation results with detailed metrics

For more information, see:
- Web interface: Access through the Streamlit app's "NWB Job Management" page
- Background processor: `u19_pipeline/automatic_job/nwb_export_handler.py`
- Utilities: `u19_pipeline/nwb_production_utils.py`